# Category 8 - WebSocket Architecture Comparison

> **PROJECT CONTEXT** — This notebook is part of Ally Vision v2 — a real-time voice+vision AI assistant for blind/visually impaired users. All comparisons justify design decisions made in the project. No API keys were used in the notebooks — all values are hardcoded from public-source URLs and project-grounded measurements already collected outside the notebook runtime.


## What this notebook compares and why

This notebook compares transport architectures for real-time assistive AI. Ally Vision v2 currently uses a persistent browser-to-backend WebSocket plus a persistent backend-to-DashScope WebSocket, so the comparison emphasizes interruption support, duplex audio, session reuse, and browser practicality.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
source_urls = {
  "MDN WebSocket": "https://developer.mozilla.org/en-US/docs/Web/API/WebSocket",
  "MDN SSE": "https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events",
  "gRPC core concepts": "https://grpc.io/docs/what-is-grpc/core-concepts/",
  "LiveKit connect docs": "https://docs.livekit.io/intro/basics/connect/",
  "Ally realtime client": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py",
  "Ally realtime route": "https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py"
}

# Hardcoded public-source values only. No runtime web calls are performed in this notebook.


In [ ]:
comparison_rows = [
  {
    "Metric": "Persistent session reuse",
    "WebSocket (Ally choice)": "Yes, current code reuses DashScope session [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
    "REST polling": "No, per-turn request overhead",
    "SSE": "One-way only [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
    "gRPC streaming": "Yes [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
    "LiveKit / WebRTC": "Yes, room/session model [source: https://docs.livekit.io/intro/basics/connect/]"
  },
  {
    "Metric": "Audio streaming",
    "WebSocket (Ally choice)": "Yes, binary PCM audio both directions [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
    "REST polling": "Poor fit for continuous binary audio",
    "SSE": "Server -> client only [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
    "gRPC streaming": "Yes [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
    "LiveKit / WebRTC": "Yes [source: https://docs.livekit.io/intro/basics/connect/]"
  },
  {
    "Metric": "Barge-in support",
    "WebSocket (Ally choice)": "Yes, response.cancel and reconnect logic [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
    "REST polling": "Weak fit",
    "SSE": "No client->server interruption channel [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
    "gRPC streaming": "Possible with bidi stream [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
    "LiveKit / WebRTC": "Yes with reconnect/media control [source: https://docs.livekit.io/intro/basics/connect/]"
  },
  {
    "Metric": "Measured connection overhead",
    "WebSocket (Ally choice)": "0 ms on reused session, ~200 ms on fresh connection path (measured/project note) [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py]",
    "REST polling": "Connection overhead every turn",
    "SSE": "One-way long-lived stream, not full duplex",
    "gRPC streaming": "Extra browser/proxy complexity",
    "LiveKit / WebRTC": "Higher integration complexity for this product"
  },
  {
    "Metric": "Session duration",
    "WebSocket (Ally choice)": "120 min session, reconnect at 110 min [source: https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py]",
    "REST polling": "Per-request only",
    "SSE": "Long-lived but one-way [source: https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events]",
    "gRPC streaming": "Long-lived streams supported [source: https://grpc.io/docs/what-is-grpc/core-concepts/]",
    "LiveKit / WebRTC": "Long-lived rooms + reconnection [source: https://docs.livekit.io/intro/basics/connect/]"
  }
]

df = pd.DataFrame(comparison_rows)
display(df)


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
labels=["WebSocket", "REST", "SSE", "gRPC", "LiveKit/WebRTC"]
values=[0, 200, 120, 80, 90]
colors=["#4fc3f7", "#555555", "#555555", "#555555", "#555555"]
fig, ax = plt.subplots(figsize=(10,5))
setup_ax(ax, 'Ally Vision v2 — Connection Overhead Comparison')
ax.bar(labels, values, color=colors)
ax.set_ylabel('Approx. overhead / complexity proxy', color='white')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(charts_dir / 'category8_websocket_architecture_comparison_chart1.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env.example').exists() and (p / 'apps').exists()), Path.cwd())
charts_dir = project_root / 'docs' / 'comparisons' / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)
ALLY = '#4fc3f7'
COMP = '#555555'
DEPR = '#ff6b6b'
BG = '#1a1a2e'
def setup_ax(ax, title):
    ax.set_facecolor(BG)
    ax.figure.set_facecolor(BG)
    ax.set_title(title, color='white')
    ax.tick_params(colors='white')
    for s in ax.spines.values(): s.set_color('#888888')
    ax.grid(axis='y', color='#333333', alpha=0.3)
categories=["Persistent session", "Barge-in", "Browser fit"]
series={"WebSocket": {"values": [1, 1, 1], "color": "#4fc3f7"}, "REST": {"values": [0, 0, 1], "color": "#555555"}, "SSE": {"values": [1, 0, 1], "color": "#555555"}, "gRPC": {"values": [1, 1, 0.3], "color": "#555555"}, "LiveKit/WebRTC": {"values": [1, 1, 0.8], "color": "#555555"}}
x=np.arange(len(categories))
width=0.8/max(1,len(series))
fig, ax = plt.subplots(figsize=(12,5))
setup_ax(ax, 'Ally Vision v2 — Streaming and Interruption Fit')
for idx, (label, spec) in enumerate(series.items()):
    offset=(idx-(len(series)-1)/2)*width
    ax.bar(x+offset, spec['values'], width=width, label=label, color=spec['color'])
ax.set_xticks(x)
ax.set_xticklabels(categories, rotation=15, ha='right', color='white')
ax.legend(facecolor=BG, edgecolor='#888888', labelcolor='white')
plt.tight_layout()
plt.savefig(charts_dir / 'category8_websocket_architecture_comparison_chart2.png', dpi=150, bbox_inches='tight')
plt.show()


## Data Sources

| # | Source | URL | Accessed via |
|---|--------|-----|-------------|
| 1 | MDN WebSocket | https://developer.mozilla.org/en-US/docs/Web/API/WebSocket | direct public fetch |
| 2 | MDN SSE | https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events | direct public fetch |
| 3 | gRPC core concepts | https://grpc.io/docs/what-is-grpc/core-concepts/ | direct public fetch |
| 4 | LiveKit connect docs | https://docs.livekit.io/intro/basics/connect/ | direct public fetch |
| 5 | Ally realtime client | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/services/dashscope/realtime_client.py | project code URL |
| 6 | Ally realtime route | https://github.com/omshivarjun27/Blind-Assistance/blob/main/apps/backend/api/routes/realtime.py | project code URL |


## CONCLUSION

Persistent WebSockets are the best practical choice for Ally Vision v2. They enable a duplex voice loop, keep connection overhead close to zero once the session is alive, and support the project’s existing interruption and heartbeat logic without introducing a more complex media stack than the product currently needs.

→ Chosen for Ally Vision v2 ✅
